# Визуализация GPX-треков

Ноутбук загружает все GPX из `cicle_gpx/`, очищает GPS-выбросы и дубли, ограничивает данные областью Санкт-Петербурга, хранит карту в `EPSG:4326` и выполняет метрические расчёты в `EPSG:32636`. Затем он показывает маршруты, профиль высоты и анализ популярности.

In [ ]:
from collections import Counter
from itertools import combinations
from math import isfinite
from pathlib import Path

import folium
from folium.plugins import Fullscreen, HeatMap, MeasureControl
import geopandas as gpd
import gpxpy
import matplotlib.pyplot as plt
import pandas as pd
from pyproj import Geod
from shapely.geometry import LineString, Point

pd.set_option("display.max_columns", 20)
GEOD = Geod(ellps="WGS84")

In [ ]:
# Работает и при запуске Jupyter из корня проекта, и из каталога notebooks/.
project_root = Path.cwd().resolve()
if not (project_root / "cicle_gpx").is_dir():
    project_root = project_root.parent

gpx_dir = project_root / "cicle_gpx"
gpx_files = sorted(gpx_dir.glob("*.gpx"))
if not gpx_files:
    raise FileNotFoundError(f"В {gpx_dir} нет GPX-файлов")

print(f"Найдено GPX-файлов: {len(gpx_files)}")
print(f"Каталог: {gpx_dir}")

In [ ]:
def geodesic_length_m(coordinates):
    """Длина линии по эллипсоиду WGS 84; coordinates имеют порядок (lon, lat)."""
    if len(coordinates) < 2:
        return 0.0
    lons, lats = zip(*coordinates)
    return abs(GEOD.line_length(lons, lats))


track_rows = []
point_rows = []

for gpx_path in gpx_files:
    with gpx_path.open(encoding="utf-8-sig") as stream:
        gpx = gpxpy.parse(stream)

    for track_index, track in enumerate(gpx.tracks, start=1):
        for segment_index, segment in enumerate(track.segments, start=1):
            points = segment.points
            if len(points) < 2:
                continue

            coordinates = [(point.longitude, point.latitude) for point in points]
            route_id = f"{gpx_path.stem}:T{track_index}:S{segment_index}"
            times = [point.time for point in points if point.time is not None]
            elevations = [point.elevation for point in points if point.elevation is not None]

            track_rows.append({
                "route_id": route_id,
                "file": gpx_path.name,
                "name": track.name or gpx_path.stem,
                "track_type": track.type,
                "points": len(points),
                "start_time": min(times) if times else pd.NaT,
                "end_time": max(times) if times else pd.NaT,
                "min_elevation_m": min(elevations) if elevations else None,
                "max_elevation_m": max(elevations) if elevations else None,
                "distance_km": geodesic_length_m(coordinates) / 1000,
                "geometry": LineString(coordinates),
            })

            cumulative_m = 0.0
            previous = None
            for point_index, point in enumerate(points):
                if previous is not None:
                    _, _, step_m = GEOD.inv(
                        previous.longitude, previous.latitude,
                        point.longitude, point.latitude,
                    )
                    cumulative_m += max(step_m, 0.0)
                point_rows.append({
                    "route_id": route_id,
                    "point_index": point_index,
                    "longitude": point.longitude,
                    "latitude": point.latitude,
                    "elevation_m": point.elevation,
                    "time": point.time,
                    "distance_km": cumulative_m / 1000,
                })
                previous = point

tracks = gpd.GeoDataFrame(track_rows, geometry="geometry", crs="EPSG:4326")
points = pd.DataFrame(point_rows)
if tracks.empty:
    raise ValueError("В GPX-файлах не найдено сегментов минимум с двумя точками")

tracks["duration_h"] = (
    pd.to_datetime(tracks["end_time"], utc=True)
    - pd.to_datetime(tracks["start_time"], utc=True)
).dt.total_seconds() / 3600
tracks["avg_speed_kmh"] = tracks["distance_km"].div(tracks["duration_h"]).where(tracks["duration_h"] > 0)

## Очистка и CRS Санкт-Петербурга

GPX хранит координаты в `EPSG:4326`. Для Санкт-Петербурга ниже явно используется метрическая проекция `EPSG:32636` — WGS 84 / UTM zone 36N. Перед анализом удаляются точки вне городской области, последовательные дубли и GPS-скачки; большие временные разрывы делят линию на отдельные фрагменты. Исходные GPX-файлы не изменяются.

In [ ]:
SOURCE_CRS = "EPSG:4326"
SPB_METRIC_CRS = "EPSG:32636"
# Прямоугольник Санкт-Петербурга с небольшим буфером, порядок: min_lon, min_lat, max_lon, max_lat.
SPB_BOUNDS_WGS84 = (29.40, 59.55, 30.90, 60.35)
MIN_POINT_DISTANCE_M = 1.0
MAX_SPEED_KMH = 80.0
MAX_TIME_GAP_SECONDS = 600
MAX_STEP_WITHOUT_TIME_M = 2_000
MIN_ELEVATION_M, MAX_ELEVATION_M = -30.0, 300.0

raw_point_count = len(points)
raw_route_count = len(tracks)
cleaning_counts = Counter()
clean_track_rows = []
clean_point_rows = []
min_lon, min_lat, max_lon, max_lat = SPB_BOUNDS_WGS84

for original_route_id, group in points.groupby("route_id", sort=False):
    metadata = tracks.loc[tracks["route_id"] == original_route_id].iloc[0]
    chunks = []
    current_chunk = []

    for point in group.sort_values("point_index").itertuples(index=False):
        lon, lat = float(point.longitude), float(point.latitude)
        in_spb = (
            isfinite(lon) and isfinite(lat)
            and min_lon <= lon <= max_lon
            and min_lat <= lat <= max_lat
        )
        if not in_spb:
            cleaning_counts["outside_or_invalid"] += 1
            if current_chunk:
                chunks.append(current_chunk)
                current_chunk = []
            continue

        if current_chunk:
            previous = current_chunk[-1]
            _, _, step_m = GEOD.inv(
                float(previous.longitude), float(previous.latitude), lon, lat
            )
            step_m = abs(step_m)
            if step_m < MIN_POINT_DISTANCE_M:
                cleaning_counts["duplicates"] += 1
                continue

            previous_has_time = pd.notna(previous.time)
            point_has_time = pd.notna(point.time)
            if previous_has_time and point_has_time:
                delta_seconds = (point.time - previous.time).total_seconds()
                if delta_seconds <= 0:
                    cleaning_counts["non_monotonic_time"] += 1
                    continue
                if delta_seconds > MAX_TIME_GAP_SECONDS:
                    chunks.append(current_chunk)
                    current_chunk = [point]
                    cleaning_counts["time_gaps"] += 1
                    continue
                speed_kmh = 3.6 * step_m / delta_seconds
                if speed_kmh > MAX_SPEED_KMH:
                    cleaning_counts["speed_spikes"] += 1
                    continue
            elif step_m > MAX_STEP_WITHOUT_TIME_M:
                chunks.append(current_chunk)
                current_chunk = [point]
                cleaning_counts["spatial_gaps"] += 1
                continue

        current_chunk.append(point)

    if current_chunk:
        chunks.append(current_chunk)

    clean_chunk_index = 0
    for chunk in chunks:
        if len(chunk) < 2:
            cleaning_counts["short_fragment_points"] += len(chunk)
            continue
        clean_chunk_index += 1
        route_id = f"{original_route_id}:C{clean_chunk_index}"
        coordinates = [(float(point.longitude), float(point.latitude)) for point in chunk]
        times = [point.time for point in chunk if pd.notna(point.time)]
        clean_elevations = []
        for point in chunk:
            elevation = point.elevation_m
            if pd.notna(elevation) and MIN_ELEVATION_M <= float(elevation) <= MAX_ELEVATION_M:
                clean_elevations.append(float(elevation))
            else:
                if pd.notna(elevation):
                    cleaning_counts["invalid_elevation"] += 1
                clean_elevations.append(None)
        valid_elevations = [value for value in clean_elevations if value is not None]

        clean_track_rows.append({
            "route_id": route_id,
            "file": metadata["file"],
            "name": metadata["name"],
            "track_type": metadata["track_type"],
            "points": len(chunk),
            "start_time": min(times) if times else pd.NaT,
            "end_time": max(times) if times else pd.NaT,
            "min_elevation_m": min(valid_elevations) if valid_elevations else None,
            "max_elevation_m": max(valid_elevations) if valid_elevations else None,
            "distance_km": geodesic_length_m(coordinates) / 1000,
            "geometry": LineString(coordinates),
        })

        cumulative_m = 0.0
        for point_index, (point, elevation) in enumerate(zip(chunk, clean_elevations)):
            if point_index:
                previous = chunk[point_index - 1]
                _, _, step_m = GEOD.inv(
                    float(previous.longitude), float(previous.latitude),
                    float(point.longitude), float(point.latitude),
                )
                cumulative_m += abs(step_m)
            clean_point_rows.append({
                "route_id": route_id,
                "point_index": point_index,
                "longitude": float(point.longitude),
                "latitude": float(point.latitude),
                "elevation_m": elevation,
                "time": point.time,
                "distance_km": cumulative_m / 1000,
            })

tracks = gpd.GeoDataFrame(clean_track_rows, geometry="geometry", crs=SOURCE_CRS)
points = pd.DataFrame(clean_point_rows)
if tracks.empty:
    raise ValueError("После очистки не осталось пригодных участков треков")

tracks_spb = tracks.to_crs(SPB_METRIC_CRS)
tracks["distance_geodesic_km"] = tracks["distance_km"]
tracks["distance_km"] = tracks_spb.length.to_numpy() / 1000
tracks_spb["distance_km"] = tracks["distance_km"].to_numpy()
tracks["duration_h"] = (
    pd.to_datetime(tracks["end_time"], utc=True)
    - pd.to_datetime(tracks["start_time"], utc=True)
).dt.total_seconds() / 3600
tracks["avg_speed_kmh"] = tracks["distance_km"].div(tracks["duration_h"]).where(tracks["duration_h"] > 0)
tracks_spb = tracks.to_crs(SPB_METRIC_CRS)  # Синхронизируем все рассчитанные атрибуты для экспорта.

cleaning_report = pd.DataFrame({
    "metric": [
        "Исходных точек", "Сохранено точек", "Удалено точек", "Исходных сегментов",
        "Вне Санкт-Петербурга / невалидные", "Дубли < 1 м",
        "Скачки скорости", "Аномалий высоты", "Разрывы времени", "Очищенных фрагментов",
    ],
    "value": [
        raw_point_count, len(points), raw_point_count - len(points), raw_route_count,
        cleaning_counts["outside_or_invalid"], cleaning_counts["duplicates"],
        cleaning_counts["speed_spikes"], cleaning_counts["invalid_elevation"],
        cleaning_counts["time_gaps"], len(tracks),
    ],
})
display(cleaning_report)
print(f"CRS карты: {tracks.crs}; метрический CRS Санкт-Петербурга: {tracks_spb.crs}")

In [ ]:
summary_columns = [
    "name", "start_time", "points", "distance_km",
    "duration_h", "avg_speed_kmh", "min_elevation_m", "max_elevation_m",
]
display(
    tracks[summary_columns]
    .sort_values("start_time", ascending=False)
    .style.format({
        "distance_km": "{:.2f}",
        "duration_h": "{:.2f}",
        "avg_speed_kmh": "{:.1f}",
        "min_elevation_m": "{:.1f}",
        "max_elevation_m": "{:.1f}",
    })
)
print(f"Суммарная длина: {tracks['distance_km'].sum():.1f} км")

In [ ]:
# Интерактивная карта всех треков.
all_routes = tracks.geometry.union_all()
center = [all_routes.centroid.y, all_routes.centroid.x]
route_map = folium.Map(location=center, tiles="OpenStreetMap", zoom_start=10, control_scale=True)
colors = ["#2563eb", "#dc2626", "#16a34a", "#9333ea", "#ea580c", "#0891b2"]

for row_index, row in tracks.reset_index(drop=True).iterrows():
    locations = [(lat, lon) for lon, lat in row.geometry.coords]
    tooltip = f"{row['name']} — {row['distance_km']:.1f} км"
    folium.PolyLine(
        locations=locations,
        color=colors[row_index % len(colors)],
        weight=4,
        opacity=0.85,
        tooltip=tooltip,
    ).add_to(route_map)

minx, miny, maxx, maxy = tracks.total_bounds
route_map.fit_bounds([[miny, minx], [maxy, maxx]])
Fullscreen().add_to(route_map)
MeasureControl(primary_length_unit="kilometers").add_to(route_map)
route_map

In [ ]:
# Выберите маршрут по его route_id. По умолчанию — самый свежий.
selected_route_id = tracks.sort_values("start_time").iloc[-1]["route_id"]
profile = points.loc[points["route_id"] == selected_route_id].dropna(subset=["elevation_m"])

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(profile["distance_km"], profile["elevation_m"], color="#2563eb", linewidth=1.5)
ax.fill_between(profile["distance_km"], profile["elevation_m"], alpha=0.15, color="#2563eb")
ax.set(title=f"Профиль высоты: {selected_route_id}", xlabel="Расстояние, км", ylabel="Высота, м")
ax.grid(alpha=0.25)
plt.show()

## Анализ популярности маршрутов

Здесь **популярность** означает повторяемость в имеющейся выборке GPX, а не популярность среди всех велосипедистов города. Пространство делится на ячейки 250×250 м в локальной метрической проекции. Каждый GPX-файл даёт ячейке не более одного голоса — это убирает перекос из-за разной частоты записи GPS-точек. Размер ячейки можно менять параметром `GRID_SIZE_M`.

In [ ]:
# Один GPX-файл считаем одной поездкой, даже если внутри несколько сегментов.
activities = (
    tracks.groupby("file", as_index=False)
    .agg(
        name=("name", "first"),
        start_time=("start_time", "min"),
        end_time=("end_time", "max"),
        distance_km=("distance_km", "sum"),
    )
)
activities["start_time"] = pd.to_datetime(activities["start_time"], utc=True)
activities["end_time"] = pd.to_datetime(activities["end_time"], utc=True)
activities["duration_h"] = (activities["end_time"] - activities["start_time"]).dt.total_seconds() / 3600
activities["avg_speed_kmh"] = activities["distance_km"].div(activities["duration_h"]).where(activities["duration_h"] > 0)

dated_activities = activities.dropna(subset=["start_time"]).copy()
dated_activities["local_time"] = dated_activities["start_time"].dt.tz_convert("Europe/Moscow")
dated_activities["month"] = dated_activities["local_time"].dt.strftime("%Y-%m")
weekday_names = {0: "Пн", 1: "Вт", 2: "Ср", 3: "Чт", 4: "Пт", 5: "Сб", 6: "Вс"}
dated_activities["weekday"] = dated_activities["local_time"].dt.dayofweek.map(weekday_names)

rides_by_month = dated_activities.groupby("month").size()
rides_by_weekday = dated_activities.groupby("weekday").size().reindex(weekday_names.values(), fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
rides_by_month.plot.bar(ax=axes[0], color="#2563eb", title="Поездки по месяцам")
rides_by_weekday.plot.bar(ax=axes[1], color="#16a34a", title="Поездки по дням недели")
for ax in axes:
    ax.set(xlabel="", ylabel="Количество поездок")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
GRID_SIZE_M = 250
local_crs = SPB_METRIC_CRS

points_metric = gpd.GeoDataFrame(
    points[["route_id", "longitude", "latitude"]].copy(),
    geometry=gpd.points_from_xy(points["longitude"], points["latitude"]),
    crs="EPSG:4326",
).to_crs(SPB_METRIC_CRS)
route_to_file = tracks.drop_duplicates("route_id").set_index("route_id")["file"]
points_metric["file"] = points_metric["route_id"].map(route_to_file)
points_metric["grid_x"] = (points_metric.geometry.x // GRID_SIZE_M).astype("int64")
points_metric["grid_y"] = (points_metric.geometry.y // GRID_SIZE_M).astype("int64")

# Один голос от одного GPX в каждой ячейке.
cell_visits = points_metric.drop_duplicates(["file", "grid_x", "grid_y"]).copy()
popularity = (
    cell_visits.groupby(["grid_x", "grid_y"], as_index=False)
    .agg(ride_count=("file", "nunique"))
)
popularity["share_pct"] = 100 * popularity["ride_count"] / activities["file"].nunique()
popularity["geometry"] = [
    Point((grid_x + 0.5) * GRID_SIZE_M, (grid_y + 0.5) * GRID_SIZE_M)
    for grid_x, grid_y in zip(popularity["grid_x"], popularity["grid_y"])
]
popularity = gpd.GeoDataFrame(popularity, geometry="geometry", crs=SPB_METRIC_CRS).to_crs(SOURCE_CRS)
popularity["longitude"] = popularity.geometry.x
popularity["latitude"] = popularity.geometry.y

top_cells = popularity.nlargest(15, "ride_count")[
    ["ride_count", "share_pct", "latitude", "longitude"]
].reset_index(drop=True)
display(top_cells.style.format({
    "share_pct": "{:.1f}%",
    "latitude": "{:.5f}",
    "longitude": "{:.5f}",
}))
print(f"Максимум: {int(popularity['ride_count'].max())} из {activities['file'].nunique()} поездок в одной ячейке")

In [ ]:
popularity_map = folium.Map(location=center, tiles="OpenStreetMap", zoom_start=11, control_scale=True)
routes_layer = folium.FeatureGroup(name="Исходные треки", show=True)
for _, row in tracks.iterrows():
    folium.PolyLine(
        [(lat, lon) for lon, lat in row.geometry.coords],
        color="#64748b", weight=2, opacity=0.25,
    ).add_to(routes_layer)
routes_layer.add_to(popularity_map)

heat_data = [
    [row.geometry.y, row.geometry.x, int(row.ride_count)]
    for _, row in popularity.iterrows()
]
HeatMap(
    heat_data, name="Популярность 250 м", radius=20, blur=16, min_opacity=0.25,
    gradient={0.2: "#3b82f6", 0.45: "#22c55e", 0.7: "#facc15", 1.0: "#dc2626"},
).add_to(popularity_map)

for _, row in popularity.nlargest(15, "ride_count").iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=min(12, 4 + row.ride_count / 2),
        color="#991b1b", fill=True, fill_opacity=0.75,
        popup=f"{int(row.ride_count)} поездок ({row.share_pct:.1f}%)",
    ).add_to(popularity_map)

minx, miny, maxx, maxy = tracks.total_bounds
popularity_map.fit_bounds([[miny, minx], [maxy, maxx]])
Fullscreen().add_to(popularity_map)
folium.LayerControl().add_to(popularity_map)
popularity_map

In [ ]:
# Сходство целых поездок по наборам посещённых ячеек.
activity_cells = {
    file_name: set(zip(group["grid_x"], group["grid_y"]))
    for file_name, group in cell_visits.groupby("file")
}
pair_rows = []
for file_a, file_b in combinations(sorted(activity_cells), 2):
    cells_a, cells_b = activity_cells[file_a], activity_cells[file_b]
    intersection = len(cells_a & cells_b)
    union = len(cells_a | cells_b)
    if intersection == 0:
        continue
    pair_rows.append({
        "route_a": file_a,
        "route_b": file_b,
        "common_cells": intersection,
        "jaccard_pct": 100 * intersection / union,
        "shorter_route_overlap_pct": 100 * intersection / min(len(cells_a), len(cells_b)),
    })

similar_routes = (
    pd.DataFrame(pair_rows)
    .sort_values(["jaccard_pct", "common_cells"], ascending=False)
    .head(15)
    .reset_index(drop=True)
)
display(similar_routes.style.format({
    "jaccard_pct": "{:.1f}%",
    "shorter_route_overlap_pct": "{:.1f}%",
}))

## Экспорт

При необходимости сохраните карту или геоданные:

```python
route_map.save(project_root / "gpx_map.html")
popularity_map.save(project_root / "gpx_popularity.html")
tracks.to_file(project_root / "tracks.geojson", driver="GeoJSON")
tracks_spb.to_file(project_root / "tracks_spb.gpkg", layer="tracks", driver="GPKG")
popularity.to_file(project_root / "popularity_cells.geojson", driver="GeoJSON")
```